In [20]:
# ============  UAE Visa RAG: Week 2 (ingest, clean, chunk, store, search, evaluate)  ============
!pip install -q chromadb sentence-transformers pymupdf rapidocr onnxruntime

import json, re
from pathlib import Path
from collections import Counter
import chromadb
from sentence_transformers import SentenceTransformer

# ---------- Settings ----------
INPUT_DIR  = "/kaggle/input"
PAGES_JSON = "/kaggle/working/pages.json"
DB_PATH    = "/kaggle/working/chroma_db"
SKIP       = ["BusinessDubai"]      # secondary source, excluded for now
MIN_SCORE  = 0.30                   # provisional "no good match" cutoff; tune in Week 4

model  = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=DB_PATH)

# ---------- 1. Load pages (from pages.json if it exists, otherwise read/OCR the PDFs) ----------
def load_pages_from_pdfs(root=INPUT_DIR):
    import pymupdf
    from rapidocr import RapidOCR
    engine = RapidOCR()

    def ocr_page(page, zoom=2):
        png = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False).tobytes("png")
        res = engine(png)
        return "\n".join(res.txts) if res.txts else ""

    pages = []
    for pdf in sorted(Path(root).rglob("*")):
        if not (pdf.is_file() and pdf.suffix.lower() == ".pdf"):
            continue
        if any(s in pdf.name for s in SKIP):
            continue
        doc, got = pymupdf.open(str(pdf)), 0
        for n, pg in enumerate(doc, start=1):
            text = pg.get_text().strip() or ocr_page(pg).strip()   # OCR only when no text layer
            if text:
                pages.append({"file": pdf.name, "page": n, "text": text})
                got += 1
        print(f"{pdf.name}: {got}/{len(doc)} pages had text")
    return pages

if Path(PAGES_JSON).exists():
    pages = json.load(open(PAGES_JSON))
    print("Loaded", len(pages), "pages from pages.json")
else:
    pages = load_pages_from_pdfs()
    json.dump(pages, open(PAGES_JSON, "w"))
    print("OCR done:", len(pages), "pages saved to pages.json")
pages = [p for p in pages if not any(s in p["file"] for s in SKIP)]

# ---------- 2. Clean (remove headers, timestamps, footer URLs, page numbers, garbled OCR lines) ----------
COMMON = {"a","i","an","as","at","be","by","if","in","is","it","of","on","or","to","up","us","we",
          "no","so","do","go","me","my","he","am","id","uk","ai","hr","ok"}
timestamp  = re.compile(r"^\d{1,2}/\d{1,2}/\d{2,4},?\s*\d{1,2}:\d{2}\s*[AP]M$")
footer_url = re.compile(r"^https?://\S+(\s+\d+/\d+)?$")
page_no    = re.compile(r"^\d+/\d+$")
header     = re.compile(r"^.{0,60}\|\s*Federal Authority for Identity, Citizenship, Customs\s*&\s*Port Security$", re.I)

def garbled(line):
    toks = re.findall(r"[A-Za-z]+", line)
    weird = sum(len(t) <= 2 and t.lower() not in COMMON for t in toks)
    return len(toks) >= 4 and weird >= 3 and weird / len(toks) > 0.25

def clean_pages(pages):
    by_file = {}
    for p in pages:
        by_file.setdefault(p["file"], []).append(p)
    out, dropped = [], Counter()
    for f, ps in by_file.items():
        counts = Counter()
        for p in ps:
            for l in set(x.strip() for x in p["text"].splitlines() if x.strip()):
                counts[l] += 1
        repeated = {l for l, c in counts.items() if c >= max(2, len(ps) // 2) and len(l) < 120}
        for p in ps:
            keep = []
            for l in (x.strip() for x in p["text"].splitlines()):
                if (not l or len(l) <= 2 or l in repeated or timestamp.match(l) or footer_url.match(l)
                        or page_no.match(l) or header.match(l) or garbled(l)):
                    if l:
                        dropped[l] += 1
                    continue
                keep.append(l)
            text = "\n".join(keep)
            if len(text.split()) >= 15:
                out.append({**p, "raw": p["text"], "text": text})
    return out, dropped

pages, dropped = clean_pages(pages)
print(f"\n{len(pages)} pages after cleaning. Lines removed (check nothing important is in here):")
for l, c in dropped.most_common(25):
    print(f"  {c:3} | {l[:95]}")

# ---------- 3. Chunk (keeps file name, page number and a document title) ----------
def title_of(fname):
    return fname.split("_10-15")[0].replace("_", " ")

def chunk_text(text, chunk_words, overlap):
    words, chunks, start = text.split(), [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_words]))
        if start + chunk_words >= len(words):
            break
        start += chunk_words - overlap
    return chunks

def make_chunks(pages, chunk_words=120, overlap=30):
    out = []
    for p in pages:
        for n, c in enumerate(chunk_text(p["text"], chunk_words, overlap)):
            out.append({"id": f'{p["file"]}|p{p["page"]}|c{n}', "text": c,
                        "file": p["file"], "page": p["page"], "title": title_of(p["file"])})
    return out

# ---------- 4. Embed and store in Chroma ----------
def build_index(chunks, name):
    try:
        client.delete_collection(name)           # always rebuild from scratch
    except Exception:
        pass
    col = client.get_or_create_collection(name=name, metadata={"hnsw:space": "cosine"})
    for i in range(0, len(chunks), 100):
        b = chunks[i:i + 100]
        col.add(
            ids=[c["id"] for c in b],
            documents=[c["text"] for c in b],
            embeddings=model.encode([f'{c["title"]}: {c["text"]}' for c in b]).tolist(),
            metadatas=[{"file": c["file"], "page": c["page"]} for c in b],
        )
    return col

# ---------- 5. Search with sources ----------
def search(question, k=5, col=None):
    col = col or collection
    res = col.query(query_embeddings=[model.encode(question).tolist()], n_results=k)
    scores = [1 - d for d in res["distances"][0]]
    print("Q:", question)
    if scores[0] < MIN_SCORE:
        print(f"   ⚠ Best score {scores[0]:.2f} is below {MIN_SCORE}: probably not in the documents.")
    for doc, meta, s in zip(res["documents"][0], res["metadatas"][0], scores):
        print(f"   {s:.2f} | {meta['file'][:48]} p.{meta['page']}")
        print("        ", doc[:200].replace("\n", " "), "...")
    print()

# ---------- 6. Evaluation: 12 in-scope + 2 out-of-scope questions ----------
tests = [
    ("What are the requirements for a green visa for self-employment?", [("GreenVisa_Selfemployment", 1), ("Green Residence Guide", 5)]),
    ("Can I sponsor my family on a green visa?",                         [("Green Residence Guide", 3)]),
    ("How long is the Golden Visa valid?",                               [("GoldenVisa", 1), ("GoldenVisa", 5)]),
    ("What documents are needed for the entrepreneur residence visa?",   [("Entrepreneur", 3)]),
    ("How much capital is needed for the Golden Visa investor category?",[("GoldenVisa", 7), ("GoldenVisa", 8)]),
    ("What is the minimum salary to sponsor a friend or relative on a visit visa?", [("Issuance_of_a_Visa", 4)]),
    ("How long is the Blue Residence valid and does it need a sponsor?", [("Blue Residence Guide", 2)]),
    ("How many years is the Green Residence valid?",                     [("Green Residence Guide", 1), ("Green Residence Guide", 3)]),
    ("How recently must I have graduated to apply for Golden Residency as an outstanding student?", [("GoldenVisa", 8), ("GoldenVisa", 9)]),
    ("Who qualifies for Golden Residency as a scientist or researcher?", [("GoldenVisa", 11), ("GoldenVisa", 5)]),
    ("What qualification and job level does an employee need for a Green Residence?", [("Green Residence Guide", 4)]),
    ("How many days does the self-employment entry visa allow before residence is issued?", [("GreenVisa_Selfemployment", 1)]),
    ("Who wins the football world cup?", None),
    ("What is the best biryani recipe?", None),
]

def evaluate(col, k=5):
    top1 = topk = n_in = out_ok = n_out = 0
    for q, expected in tests:
        res = col.query(query_embeddings=[model.encode(q).tolist()], n_results=k)
        scores = [1 - d for d in res["distances"][0]]
        if expected is None:
            n_out += 1
            out_ok += scores[0] < MIN_SCORE
        else:
            n_in += 1
            hits = [any(s in m["file"] and m["page"] == pg for s, pg in expected) for m in res["metadatas"][0]]
            top1 += hits[0]
            topk += any(hits)
    return top1, topk, n_in, out_ok, n_out

# ---------- 7. Compare chunk sizes, keep the best one ----------
results = []
print("\n| Chunk size / overlap | Chunks | Top-1 | Top-5 | Out-of-scope rejected |")
print("|---|---|---|---|---|")
for size, ov in [(60, 15), (120, 30), (200, 40)]:
    chunks = make_chunks(pages, size, ov)
    col = build_index(chunks, f"uae_visa_{size}")
    t1, tk, n_in, ook, n_out = evaluate(col)
    results.append((tk, t1, size, ov))
    print(f"| {size} / {ov} | {len(chunks)} | {t1}/{n_in} | {tk}/{n_in} | {ook}/{n_out} |")

best_tk, best_t1, best_size, best_ov = max(results)
chunks = make_chunks(pages, best_size, best_ov)
collection = build_index(chunks, "uae_visa")
print(f"\nBest setting: {best_size} words / {best_ov} overlap  ->  final collection 'uae_visa' with {collection.count()} chunks")

# ---------- 8. Demo searches ----------
print("\n" + "=" * 70 + "\n")
for q in ["What are the requirements for a green visa for self-employment?",
          "Can I sponsor my family on a green visa?",
          "How long is the Golden Visa valid?",
          "Who wins the football world cup?"]:
    search(q)

# ---------- 9. Fact check: which salary/income figures actually appear in the official PDFs? ----------
pattern = re.compile(r"15,?000|30,?000|salary|monthly income", re.I)
print("=" * 70 + "\nSalary / income lines found in the documents (first 25):")
shown = 0
for p in pages:
    for line in p["raw"].splitlines():
        if pattern.search(line) and shown < 25:
            print(f'  {p["file"][:38]} p.{p["page"]}: {line.strip()[:110]}')
            shown += 1

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 45 pages from pages.json

44 pages after cleaning. Lines removed (check nothing important is in here):
   14 | Golden Residency Guide | Federal Authority for Identity, Citizenship, Customs & Port Security
   14 | https://icp.gov.ae/en/services/uae-golden-residency/
   13 | 10/15/26, 2:43 PM
   10 | V
    8 | https://icp.gov.ae/en/services/uae-blue-residency/
    7 | 自
    7 | https://icp.gov.ae/en/services/uae-green-residency/
    7 | Issuance of a Visa
    6 | 品
    6 | ★
    6 | Blue Residence Guide | Federal Authority for Identity, Citizenship, Customs & Port Security
    6 | 10/15/26, 2:44 PM
    6 | Green Residence Guide | Federal Authority for Identity, Citizenship, Customs & Port Security
    6 | https://icp.gov.ae/en/services-details/?serviceid=64afe3c1035448005bd52e60
    5 | FEDERAL AUTHORITY FOR IDENTITY, CITIZENSHIP, CUSTOMS & PORT SECURITY
    5 | GOVERNMENT OF DUBAI
    5 | PROUD OF UAE
    5 | GENERAL DIRECTORATE FOR IDENTITY AND FOREIGNERS AFFAIRS - DUBAI
    5 |